# Examen bimestral corregido
**Introducción a los Sistemas de Información**  
**Caso de estudio: Papelería universitaria**

En esta versión corregí los errores de código y completé las preguntas que faltaban. También agregué anotaciones breves para explicar qué estaba haciendo mal y cómo lo corregí.

In [1]:
import pandas as pd
from pathlib import Path

# Busco los archivos en la carpeta del notebook. También considero los nombres
# que tenían los archivos que me entregaron.
def buscar_archivo(nombres):
    for nombre in nombres:
        ruta = Path(nombre)
        if ruta.exists():
            return ruta
    raise FileNotFoundError(f"No encontré ninguno de estos archivos: {nombres}")

ruta_ventas = buscar_archivo(["ventas.csv", "/mnt/data/ventas.csv"])
ruta_productos = buscar_archivo(["productos.csv", "productos_.csv", "/mnt/data/productos_.csv"])
ruta_clientes = buscar_archivo(["clientes.csv", "/mnt/data/clientes.csv"])

ventas = pd.read_csv(ruta_ventas, parse_dates=["fecha"])
productos = pd.read_csv(ruta_productos)
clientes = pd.read_csv(ruta_clientes)

print("Archivos cargados correctamente")
print("Ventas:", ventas.shape)
print("Productos:", productos.shape)
print("Clientes:", clientes.shape)

Archivos cargados correctamente
Ventas: (2100, 9)
Productos: (400, 4)
Clientes: (100, 4)


## 1. Líneas, tickets y unidades

**Mi corrección:** Antes solo mostré las columnas `id_linea` e `id_ticket`, pero no calculé los conteos con métodos de pandas y me faltó contar las unidades. Ahora uso `count()`, `nunique()` y la suma de `cantidad`, porque cada fila representa una línea de venta.

In [2]:
lineas = ventas["id_linea"].count()
tickets = ventas["id_ticket"].nunique()
unidades = ventas["cantidad"].sum()

print("Líneas de venta:", lineas)
print("Tickets distintos:", tickets)
print("Unidades vendidas:", unidades)

# Una línea es un registro de producto dentro de un ticket;
# un ticket puede tener varias líneas; las unidades suman las cantidades.

Líneas de venta: 2100
Tickets distintos: 300
Unidades vendidas: 3880


## 2. Tickets por sucursal

**Mi corrección:** La idea de agrupar por sucursal era correcta, pero usé `ventas` sin asegurar que esa variable estuviera cargada. Ahora trabajo con el DataFrame que cargué al inicio y cuento tickets distintos para no contar varias veces un mismo ticket.

In [3]:
tickets_sucursal = ventas.groupby("sucursal")["id_ticket"].nunique().sort_values(ascending=False)
print("TICKETS POR SUCURSAL")
print(tickets_sucursal)
print("\nSucursal con más tickets:", tickets_sucursal.idxmax())
print("Cantidad:", tickets_sucursal.max())

TICKETS POR SUCURSAL
sucursal
Central    146
Norte       83
Sur         71
Name: id_ticket, dtype: int64

Sucursal con más tickets: Central
Cantidad: 146


## 3. Métodos de pago

**Mi corrección:** Había contado los métodos con `value_counts()`, pero no incluí explícitamente los valores vacíos y el código anterior produjo un error porque la variable `ventas` no estaba definida. Primero dejo una sola fila por ticket y método de pago, y uso `dropna=False` para mostrar los datos sin registrar.

In [4]:
tickets_pago = ventas[["id_ticket", "metodo_pago"]].drop_duplicates()

conteo_pago = tickets_pago["metodo_pago"].value_counts(dropna=False)
conteo_pago.index = ["Sin registrar" if pd.isna(x) else x for x in conteo_pago.index]

print("TICKETS POR MÉTODO DE PAGO")
print(conteo_pago)

TICKETS POR MÉTODO DE PAGO
Efectivo         144
Tarjeta           84
Transferencia     48
Sin registrar     24
Name: count, dtype: int64


## 4. Precios de lista por categoría

**Mi corrección:** Esta agrupación estaba bien planteada. La mantengo, pero me aseguro de usar el catálogo `productos`, porque la pregunta pide el precio de lista y no el precio unitario de las ventas.

In [5]:
precios_categoria = productos.groupby("categoria")["precio_lista"].agg(
    minimo="min",
    maximo="max",
    promedio="mean"
)

print("PRECIOS DE LISTA POR CATEGORÍA")
display(precios_categoria.round(2))

PRECIOS DE LISTA POR CATEGORÍA


,minimo,maximo,promedio
categoria,,,
Accesorios,0.40,22.90,1.62
Archivo,0.35,3.95,1.67
Escritura,0.40,3.75,1.35
Papel y cuadernos,0.28,5.75,1.67


## 5. Unidades por sucursal y categoría

**Mi corrección:** La tabla cruzada estaba bien encaminada. La corrección importante es asegurar la relación entre ventas y productos mediante `id_producto`, y comprobar que el total de la tabla sea igual a la suma de `cantidad` en ventas.

In [6]:
ventas_productos = ventas.merge(
    productos[["id_producto", "categoria"]],
    on="id_producto",
    how="left",
    validate="many_to_one"
)

tabla_unidades = pd.pivot_table(
    ventas_productos,
    index="sucursal",
    columns="categoria",
    values="cantidad",
    aggfunc="sum",
    fill_value=0,
    margins=True,
    margins_name="Total"
)

print("UNIDADES POR SUCURSAL Y CATEGORÍA")
display(tabla_unidades)

total_archivo = ventas["cantidad"].sum()
total_tabla = tabla_unidades.loc["Total", "Total"]

print("Total de unidades en ventas.csv:", total_archivo)
print("Total general de la tabla:", total_tabla)
print("¿Coinciden?:", total_archivo == total_tabla)

UNIDADES POR SUCURSAL Y CATEGORÍA


categoria,Accesorios,Archivo,Escritura,Papel y cuadernos,Total
sucursal,,,,,
Central,453,481,437,524,1895
Norte,256,264,306,259,1085
Sur,191,223,238,248,900
Total,900,968,981,1031,3880


Total de unidades en ventas.csv: 3880
Total general de la tabla: 3880
¿Coinciden?: True


## 6. Importe de ventas por sucursal

**Mi corrección:** El cálculo de `cantidad × precio_unitario` era correcto, pero el código fallaba porque `ventas` no estaba definida en ese momento. Ahora uso el DataFrame cargado al inicio y presento los importes con dos decimales.

In [7]:
ventas["importe_linea"] = ventas["cantidad"] * ventas["precio_unitario"]

importe_sucursal = ventas.groupby("sucursal")["importe_linea"].sum().sort_values(ascending=False)

print("IMPORTE DE VENTAS POR SUCURSAL")
display(importe_sucursal.to_frame("Importe total").style.format("${:,.2f}"))

IMPORTE DE VENTAS POR SUCURSAL


,Importe total
sucursal,
Central,"$3,025.04"
Norte,"$1,635.35"
Sur,"$1,443.24"


## 7. Tamaño de las compras

**Mi corrección:** La clasificación estaba correcta, porque primero sumaba las cantidades de todas las líneas de cada ticket. La dejo más clara y ordeno las categorías para que aparezcan en el orden solicitado.

In [8]:
unidades_ticket = ventas.groupby("id_ticket")["cantidad"].sum()

def clasificar_compra(unidades):
    if unidades <= 7:
        return "Hasta 7 unidades"
    elif unidades <= 14:
        return "De 8 a 14 unidades"
    else:
        return "15 o más unidades"

categoria_ticket = unidades_ticket.apply(clasificar_compra)
orden = ["Hasta 7 unidades", "De 8 a 14 unidades", "15 o más unidades"]
conteo_tamanos = categoria_ticket.value_counts().reindex(orden, fill_value=0)

print("TICKETS SEGÚN EL TAMAÑO DE LA COMPRA")
print(conteo_tamanos)

TICKETS SEGÚN EL TAMAÑO DE LA COMPRA
cantidad
Hasta 7 unidades       39
De 8 a 14 unidades    153
15 o más unidades     108
Name: count, dtype: int64


## 8. Ventas por mes

**Mi corrección:** En mi notebook original el enunciado de esta pregunta estaba repetido: aparecía nuevamente la pregunta 7. Además, convertía `fecha` a periodo mensual sin haber indicado que debía leerse como fecha, y dependía de `importe_linea` calculado en otra celda. Ahora corrijo el enunciado, convierto la fecha al cargar los datos y calculo el importe por línea antes de agrupar.

In [9]:
ventas["mes"] = ventas["fecha"].dt.to_period("M")

ventas_mes = ventas.groupby("mes").agg(
    tickets=("id_ticket", "nunique"),
    importe_total=("importe_linea", "sum")
)

print("TICKETS E IMPORTE DE VENTAS POR MES")
display(ventas_mes.style.format({"importe_total": "${:,.2f}"}))

mes_mayor = ventas_mes["importe_total"].idxmax()
print("Mes con mayor importe:", mes_mayor)
print("Importe:", f"${ventas_mes.loc[mes_mayor, 'importe_total']:,.2f}")

TICKETS E IMPORTE DE VENTAS POR MES


,tickets,importe_total
mes,,
2025-11,25,$492.51
2025-12,25,$645.54
2026-01,25,$447.48
2026-02,25,$466.36
2026-03,25,$595.33
2026-04,25,$505.43
2026-05,25,$453.94
2026-06,25,$500.54
2026-07,25,$499.88


Mes con mayor importe: 2025-12
Importe: $645.54


## 9. Producto más vendido por sucursal

**Mi corrección:** Esta pregunta no estaba desarrollada en mi notebook. Para resolverla, relaciono las ventas con el catálogo de productos, sumo las unidades por sucursal y producto, y selecciono el máximo de cada sucursal. Así también conservo todos los productos si existe un empate.

In [10]:
ventas_detalle = ventas.merge(
    productos[["id_producto", "producto"]],
    on="id_producto",
    how="left",
    validate="many_to_one"
)

unidades_producto = ventas_detalle.groupby(
    ["sucursal", "id_producto", "producto"], as_index=False
)["cantidad"].sum()

maximos_sucursal = unidades_producto.groupby("sucursal")["cantidad"].transform("max")
productos_mas_vendidos = unidades_producto[
    unidades_producto["cantidad"] == maximos_sucursal
].sort_values(["sucursal", "id_producto"])

print("PRODUCTO(S) MÁS VENDIDO(S) POR SUCURSAL")
display(productos_mas_vendidos.rename(columns={
    "id_producto": "Código",
    "producto": "Producto",
    "cantidad": "Unidades vendidas"
}))

PRODUCTO(S) MÁS VENDIDO(S) POR SUCURSAL


,sucursal,Código,Producto,Unidades vendidas
376,Central,P399,Estuche escolar (gris),17
488,Norte,P141,Marcador de punta fina (naranja),17
697,Sur,P016,Cuaderno cuadriculado A5 (24 hojas),11
923,Sur,P304,Portadocumentos con cierre (rojo),11


## 10. Compras por tipo de cliente

**Mi corrección:** Esta pregunta tampoco estaba desarrollada. Uno las ventas con los clientes usando `ventas.id_cliente` y `clientes.id`. Después calculo clientes distintos, tickets distintos e importe total por tipo de cliente. Uso `nunique()` para evitar contar varias veces al mismo cliente o ticket.

In [11]:
ventas_clientes = ventas.merge(
    clientes[["id", "tipo_cliente"]],
    left_on="id_cliente",
    right_on="id",
    how="left",
    validate="many_to_one"
)

resumen_clientes = ventas_clientes.groupby("tipo_cliente").agg(
    clientes_distintos=("id_cliente", "nunique"),
    tickets=("id_ticket", "nunique"),
    importe_total=("importe_linea", "sum")
).sort_index()

print("COMPRAS POR TIPO DE CLIENTE")
display(resumen_clientes.style.format({"importe_total": "${:,.2f}"}))

COMPRAS POR TIPO DE CLIENTE


,clientes_distintos,tickets,importe_total
tipo_cliente,,,
Docente,16,46,$899.74
Estudiante,72,226,"$4,688.00"
Personal administrativo,12,28,$515.89


## Anotación final

Corregí los errores de carga y uso de variables, completé los cálculos que faltaban y añadí las preguntas 9 y 10. También corregí el enunciado duplicado de la pregunta 8. Los resultados se generan directamente desde los tres archivos CSV.